
# AY15 - Ejercicios de Networking y Arquitectura Cliente-Servidor (IIC2233)

Recomendación: resuelve cada pregunta en papel, con tiempo (P1–P3: 25 min c/u; P4 y P5: 50 min c/u; P6: 25 min), y recién después revisa la pauta.

## Pregunta 1. Misceláneo [6 puntos]

**Parte A. JSON (1.5 puntos)**

A.1) Indica qué imprime el siguiente código y explica en máximo 2 líneas cada diferencia entre `datos` y `copia`.

```python
import json

datos = {1: ("Dune", "Herbert"), "disponible": True, "dueño": None}
copia = json.loads(json.dumps(datos))
print(copia)
print(datos == copia)
```

A.2) Considera este *hook*. Escribe, en orden, cada diccionario que recibe `contar` y el valor final de `resultado`.

```python
def contar(diccionario):
    return len(diccionario)

resultado = json.loads('{"a": {"b": {}, "c": [1, {"d": 0, "e": 0}]}, "f": 2}', object_hook=contar)
```

**Parte B. pickle (1.5 puntos)**

Un `Jugador` guarda su `nombre`, su `puntaje` y `self.socket`, el socket con el que está conectado al servidor. El servidor guarda el estado de la partida con `pickle.dump` cada minuto.

B.1) Explica en máximo 2 líneas qué problema aparece al serializar un `Jugador` y por qué el atributo `socket` no tiene sentido en otra ejecución del programa.

B.2) Escribe `__getstate__` y `__setstate__` de modo que se guarde todo excepto el socket, y que al cargar el jugador quede con `socket = None` y `conectado = False`, sin modificar el objeto original al serializar.

B.3) Un ayudante propone que los clientes envíen su estado al servidor con `pickle.dumps` por el socket. Explica en máximo 2 líneas el riesgo y qué formato usarías en su lugar.

**Parte C. HTTP y `requests` (1 punto)**

C.1) Para la URL `https://api.dccanje.cl:8443/libros/42/resenas?pagina=2&orden=fecha#comentarios`, indica el host, el puerto, la ruta, los parámetros de consulta y qué parte **no** llega al servidor.

C.2) Clasifica cada respuesta como éxito, error del cliente o error del servidor, y di qué significa en una frase: `201`, `204`, `401`, `404`, `405`, `422`, `429`, `503`.

**Parte D. Transporte y sockets (2 puntos)**

D.1) Un juego de carreras manda la posición de cada auto 30 veces por segundo; un chat manda mensajes que no pueden perderse ni llegar desordenados. Elige TCP o UDP para cada uno y justifica en máximo 2 líneas.

D.2) Un servidor hace `datos = sock.recv(4096)` y el cliente envió un mensaje de 3000 bytes con `sendall`. Explica en máximo 2 líneas por qué `datos` puede tener menos de 3000 bytes, y qué significa que `recv` retorne `b""`.

D.3) Explica en máximo 3 líneas la diferencia entre `send` y `sendall`, y qué información se pierde si `sendall` lanza una excepción a mitad del envío.

## Pregunta 2. Protocolo y manejo de bytes [6 puntos]

El chat de la aplicación DCCanje envía textos por un socket TCP con el siguiente protocolo:

1. El texto se codifica en **UTF-8**; esos bytes son el *payload*.
2. El payload se divide en *chunks* de **6 bytes**. Si el último chunk queda incompleto, se rellena con bytes `00` hasta completar 6.
3. Cada chunk se envía precedido de **1 byte** con su número de chunk, partiendo en 1, en big endian.
4. Antes de todo se envía un *header* de **3 bytes**: 2 bytes con el largo real del payload (sin relleno) en **little endian**, y 1 byte con la cantidad de chunks.

```text
[ largo (2 bytes) ][ n° chunks (1 byte) ][ n° 1 (1 byte) ][ chunk 1 (6 bytes)][ n° 2 (1 byte) ][ chunk 2 (6 bytes) ] ...
```

Puedes usar esta tabla UTF-8: `H`=48, `o`=6F, `l`=6C, `a`=61, `,`=2C, espacio=20, `n`=6E, `d`=64, `ñ`=C3 B1, `ú`=C3 BA, `¿`=C2 BF, `L`=4C, `i`=69, `b`=62, `r`=72, `?`=3F.

**2.1) (2 puntos)** Para el texto `Hola, ñandú`:

- ¿Cuántos caracteres tiene el texto y cuántos bytes tiene el payload? Explica la diferencia.
- Escribe en hexadecimal el header completo.
- Escribe en hexadecimal el último bloque enviado (número de chunk + chunk).
- ¿Cuántos bytes viajan en total por el socket?

**2.2) (2 puntos)** El receptor recibe exactamente estos bytes:

```text
08 00 02 01 C2 BF 4C 69 62 72 02 6F 3F 00 00 00 00
```

Indica el largo del payload, la cantidad de chunks y el texto recibido. Explica por qué el receptor necesita el largo real, si igual conoce la cantidad de chunks.

**2.3) (2 puntos)** Escribe en pseudocódigo `recibir_texto(sock) -> str`, que lea **un** mensaje completo con este protocolo. Considera que `recv(n)` puede entregar menos de `n` bytes y que, si la conexión se cierra a mitad del mensaje, debes levantar `ConnectionError`. Debes verificar que los números de chunk lleguen en orden.

## Pregunta 3. Servicios web: cliente y servidor [6 puntos]

DCCanje expone una API Flask para administrar los libros publicados para intercambio. Cada libro es un diccionario `{"id": int, "titulo": str, "autor": str, "estado": "disponible" | "reservado"}` y todos se guardan en el diccionario global `LIBROS`, cuya llave es el id. El servidor se ejecuta con `threaded=True`.

| Método y ruta | Qué hace |
| --- | --- |
| `GET /libros?autor=<str>&limite=<int>` | Lista los libros del autor (todos si no se indica), hasta `limite` (por defecto 10) |
| `POST /libros` | Crea un libro a partir de un JSON con `titulo` y `autor` (ambos `str` no vacíos) y responde el libro creado |
| `POST /libros/<id>/reservar` | Reserva un libro disponible |
| `DELETE /libros/<id>` | Elimina el libro |

**3.1) (2 puntos)** Escribe en pseudocódigo los endpoints `GET /libros` y `POST /libros`. Responde `400` si el cuerpo no es un objeto JSON o le faltan llaves, `422` si los valores no cumplen el formato, y `201` con el libro creado si todo está bien.

**3.2) (1 punto)** Escribe la función de cliente `buscar_por_autor(autor: str) -> list` que consulte `GET /libros` con `requests`, pasando el autor y un límite de 5. La función debe evitar esperas infinitas, retornar `[]` si el servidor responde un error o no responde, y no asumir que el cuerpo es JSON sin revisarlo.

**3.3) (1.5 puntos)** Completa la tabla con el código de estado que retorna **tu** implementación, y justifica cada caso en una línea. Supone que existe el libro 7, disponible.

| Solicitud | Código |
| --- | --- |
| `POST /libros` con `json={"titulo": "Dune"}` |  |
| `POST /libros` con `json={"titulo": "", "autor": "Herbert"}` |  |
| `POST /libros` con `data={"titulo": "Dune", "autor": "Herbert"}` |  |
| `GET /libros/7` |  |
| `PUT /libros` |  |
| `DELETE /libros/abc` (la ruta usa `<int:id>`) |  |

**3.4) (1.5 puntos)** El endpoint de reserva está implementado así:

```python
@app.route("/libros/<int:id_libro>/reservar", methods=["POST"])
def reservar(id_libro):
    libro = LIBROS.get(id_libro)
    if libro is None:
        abort(404)
    if libro["estado"] == "disponible":
        registrar_auditoria(id_libro)   # tarda ~50 ms
        libro["estado"] = "reservado"
        return {"reservado": True}
    return {"reservado": False, "motivo": "ya reservado"}, 409
```

- Dos usuarios envían la reserva del libro 7 casi al mismo tiempo y **ambos** reciben `{"reservado": true}`. Explica la secuencia de ejecución que lo produce y corrígela.
- Un usuario hace doble clic en “Reservar” y su cliente envía dos `POST`. Propón una solución en el cliente y una en el servidor, y explica por qué la del cliente sola no basta.
- Si en producción la API corre en 4 procesos, ¿sigue funcionando tu corrección? Justifica en máximo 2 líneas.

## Pregunta 4. DCCTrivia: GUI, back-end y servidor [50 puntos]

DCCTrivia es un juego de preguntas en tiempo real. Un **servidor TCP sin interfaz gráfica** coordina la partida y cada jugador usa un **cliente PyQt6** separado en *front-end* y *back-end*.

**La partida.** Cuando hay al menos 2 jugadores conectados, el servidor envía a todos una pregunta con 4 opciones (A, B, C, D) y un plazo de **15 segundos**. Cada jugador puede responder **una sola vez** por pregunta.

- La **primera** respuesta correcta recibe 3 puntos; las demás correctas, 1 punto; las incorrectas, 0.
- Las respuestas que llegan al servidor después del plazo se rechazan.
- La pregunta se cierra cuando todos respondieron o cuando vence el plazo, **aunque nadie envíe mensajes**. Al cerrarse, el servidor envía a todos el resultado (opción correcta y tabla de puntajes), espera 5 segundos y envía la siguiente pregunta.
- Los jugadores también pueden escribir en un chat que el servidor reenvía a todos.

**Mensajes.** Todo mensaje es un diccionario serializado con JSON y codificado en UTF-8, precedido por 4 bytes con su largo en big endian. El cliente envía `UNIRSE(nombre)`, `RESPONDER(id_pregunta, opcion)` y `CHAT(texto)`. El servidor envía `PREGUNTA(id_pregunta, texto, opciones, duracion)`, `RESPUESTA_RECIBIDA(id_pregunta, aceptada, motivo)`, `RESULTADO(id_pregunta, correcta, puntajes)`, `CHAT(autor, texto)` y `JUGADOR_SALIO(nombre)`. Puedes agregar mensajes o llaves, justificando cada agregado.

**El cliente.** La ventana muestra la pregunta, 4 botones, una **cuenta regresiva** que se actualiza cada 0,1 s, la tabla de puntajes y el chat. Al responder, los 4 botones se deshabilitan hasta la siguiente pregunta. Si se pierde la conexión, la ventana lo informa y deshabilita todo, sin congelarse ni cerrarse.

> **Supuestos.** Puedes hacer los supuestos que necesites si no contradicen el enunciado. Indícalos explícitamente.

**4.1) (10 pts) Diagrama del sistema.** Dibuja el servidor y dos clientes. Dentro del servidor, muestra sus threads. Dentro de cada cliente, muestra el front-end, el back-end y el thread que escucha. En cada flecha indica qué viaja y en qué formato: mensajes por socket entre clientes y servidor, y **señales** (nombre y argumentos) entre front-end, back-end y thread.

**4.2) (10 pts) Threads y timers.** Para el servidor y para el cliente, haz una tabla con cada `Thread`, `QThread`, `QTimer` o `threading.Timer` de tu diseño: qué hace, cuándo se inicia, qué thread llama a su `start()` y por qué elegiste ese mecanismo y no otro. Responde además:

- ¿Por qué la cuenta regresiva del cliente no debería ser un thread con `sleep(0.1)`, y por qué el plazo de 15 s del servidor no puede ser un `QTimer`?
- ¿Qué recursos comparten los threads del servidor y cómo proteges cada uno?
- ¿Necesita algún lock el cliente? Justifica a partir de la regla de oro.

**4.3) (14 pts) Diagrama de señales.** Ana y Bruno están jugando la pregunta 7, cuya respuesta correcta es B. Bruno ya respondió C. Ana presiona el botón **B** a los 4 segundos y es la última en responder. Dibuja un **diagrama de secuencia** desde el clic de Ana hasta que la tabla de puntajes se actualiza en la pantalla de Bruno. Para cada paso indica:

- quién emite y quién recibe (objeto y método);
- si es una señal de PyQt (nombre y argumentos), un mensaje por socket (tipo y contenido) o una llamada directa;
- **en qué thread se ejecuta** cada método;
- qué locks se toman en el servidor y cuándo se sueltan, y qué pasa con el plazo de 15 s de la pregunta;
- qué pasa con la cuenta regresiva de cada cliente.

**4.4) (8 pts) Análisis de código.** Un grupo entregó este cliente. Las funciones `enviar(sock, dict)` y `recibir(sock) -> dict` implementan correctamente el protocolo. Encuentra **al menos 6 errores** de diseño o de PyQt. Para cada uno indica la línea, qué síntoma produciría al ejecutar y cómo lo corriges.

```python
class EscucharThread(QThread):
    def __init__(self, sock, ventana):
        super().__init__()
        self.sock = sock
        self.ventana = ventana

    def run(self):
        while True:
            mensaje = recibir(self.sock)
            if mensaje["tipo"] == "PREGUNTA":
                self.ventana.label_pregunta.setText(mensaje["texto"])
                self.timer = QTimer()
                self.timer.timeout.connect(self.ventana.descontar)
                self.timer.start(100)


class Logica(QObject):
    señal_resultado = pyqtSignal(dict)

    def conectar(self, ventana):
        self.sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
        self.sock.connect((HOST, PORT))
        hilo = EscucharThread(self.sock, ventana)
        hilo.start()

    def enviar_respuesta(self, opcion):
        enviar(self.sock, {"tipo": "RESPONDER", "opcion": opcion})
        respuesta = recibir(self.sock)
        self.señal_resultado.emit(respuesta)


class Ventana(QWidget):
    senal_responder = pyqtSignal(str)

    def __init__(self):
        super().__init__()
        # ... crea label_pregunta y los botones ...
        self.boton_a.clicked.connect(self.responder("A"))

    def closeEvent(self, evento):
        evento.accept()
```

**4.5) (8 pts) Situaciones.** Explica qué ocurre en tu diseño y por qué.

1. (2 pts) El servidor se cae mientras Ana mira la pregunta 8. Describe la cadena desde `recv` hasta lo que ve Ana.
2. (2 pts) Ana cierra su ventana a mitad de la partida. ¿Qué debe hacer su cliente antes de terminar, y qué hace el servidor con su thread, su socket y la tabla de puntajes?
3. (2 pts) Ana y Bruno envían la respuesta correcta con 5 ms de diferencia. ¿Quién recibe 3 puntos y qué garantiza que no reciban 3 puntos ambos? ¿Qué pasa si el plazo vence mientras el servidor procesa la respuesta de Ana?
4. (2 pts) La cuenta regresiva de Ana muestra 0,2 s, Ana responde, y el servidor la rechaza con `motivo: "fuera de plazo"`. ¿Es un error del diseño? Justifica quién debe decidir el plazo y por qué.

**Bonus (+2 pts).** El servidor envía `{"tipo": "CHAT", "autor": "Ana", "texto": "¡Sí!"}` con `json.dumps(...).encode("utf-8")`. Indica cuántos bytes tiene el payload y escribe en hexadecimal los 4 bytes del header. Ojo con cómo `json.dumps` escribe los caracteres no ASCII por defecto.

## Pregunta 5. DCCSubastas: un cliente, dos servidores [50 puntos]

DCCSubastas permite pujar en vivo por libros usados. Hay **dos servidores TCP independientes**, en puertos distintos:

- **Servidor de subastas (puerto 5000).** Mantiene varias subastas abiertas al mismo tiempo. Cada subasta tiene un id, un libro, la puja más alta, el usuario que la hizo y una hora de cierre. Recibe `PUJAR(id_subasta, monto)`. Una puja es válida si supera la más alta en al menos $500 y la subasta sigue abierta. Si una puja válida llega en los **últimos 10 segundos**, el cierre se extiende 10 segundos desde ese momento. Al cerrar, la subasta queda adjudicada aunque nadie envíe mensajes, y se agrega una línea a `adjudicaciones.csv`.
- **Servidor de chat (puerto 5001).** Reenvía los mensajes de chat de cada subasta a quienes la están mirando.

**Saldo.** Antes de aceptar una puja, el servidor de subastas consulta la API externa **DCCBanco** (`GET /saldo/<usuario>`, responde `{"saldo": int}`). DCCBanco tarda entre 0,1 y 6 segundos, y a veces responde `503` o no responde.

**Difusión.** Cada puja aceptada se difunde como `NUEVA_PUJA(id_subasta, monto, usuario, cierre)` a todos los clientes conectados al servidor de subastas, y cada cierre como `SUBASTA_CERRADA(id_subasta, ganador, monto)`. Algunos clientes tienen mala conexión y leen muy lento: un `sendall` hacia ellos puede quedar bloqueado varios segundos.

**API web.** El servidor de subastas expone en el mismo proceso `GET /subastas/<id>/historial` (Flask), que responde en JSON todas las pujas aceptadas de esa subasta.

**Formato.** Ambos servidores usan diccionarios serializados con JSON, en UTF-8, con un header de 4 bytes big endian con el largo.

> **Supuestos.** Indica explícitamente cada supuesto. Lo agregado sin justificación no se considera.

**5.1) (12 pts)**

- (6 pts) Dibuja un diagrama con el cliente (front-end, back-end y sus threads), ambos servidores, DCCBanco, la API, `adjudicaciones.csv` y un navegador que consulta el historial. En cada flecha indica qué viaja y en qué formato.
- (6 pts) Describe los threads del **cliente**: cuántos sockets tiene, cuántos threads escuchan y por qué **no** puede usar un solo thread que haga `recv` alternadamente en ambos sockets. Indica cómo el back-end distingue de qué servidor vino cada mensaje.

**5.2) (12 pts)** Describe los threads del **servidor de subastas** (qué hace cada uno, cuándo se inicia, quién llama a su `start()`) y sus recursos compartidos con el mecanismo que protege cada uno. Compara dos formas de cerrar las subastas a tiempo y elige una, justificando:

- un `threading.Timer` por subasta, que hay que reemplazar cada vez que el cierre se extiende;
- un único thread vigilante que revisa todas las subastas cada 0,5 s.

**5.3) (16 pts)** Pedro mira la subasta S-3 (puja más alta $8.000, cierre en 6 s) y envía `PUJAR("S-3", 9000)`. DCCBanco responde a los 2 s con saldo suficiente. Relata paso a paso, indicando el thread de cada paso, qué ocurre desde que el servidor empieza a recibir el mensaje hasta que Rosa, que tiene una conexión lenta, ve la nueva puja en su pantalla. Tu relato debe responder:

- qué se verifica antes de llamar a DCCBanco, qué se **vuelve a verificar** después, y por qué;
- qué lock se toma y si está tomado durante la llamada a DCCBanco;
- cómo cambia el cierre de S-3 (la puja llega con 4 s restantes al momento de aceptarse);
- cómo se difunde la puja sin que la conexión lenta de Rosa retrase a Pedro ni a los demás;
- en el cliente de Rosa, qué thread recibe, qué señales se emiten y en qué thread se actualiza la pantalla.

**5.4) (10 pts)** Explica qué ocurre en tu diseño.

1. (3 pts) Mientras DCCBanco tarda 5 s en responder a Pedro, Quena puja $9.500 por S-3 y su saldo se confirma en 0,2 s.
2. (3 pts) DCCBanco no responde nunca a la consulta de Pedro.
3. (2 pts) El servidor de chat se cae, pero el de subastas sigue funcionando. ¿Qué ve el usuario y qué sigue funcionando?
4. (2 pts) Un navegador consulta el historial de S-3 justo mientras se acepta la puja de Pedro. ¿Puede recibir un historial a medio escribir? Justifica.

## Pregunta 6. Análisis de código [6 puntos]

**6.1) (2 puntos)** Escribe el output del siguiente código. Supone que los threads se ejecutan sin retrasos adicionales a las esperas indicadas.

```python
import threading
import time

lock = threading.Lock()
evento = threading.Event()

def servidor():
    print("S-escuchando")
    evento.wait()
    with lock:
        print("S-procesa")
        time.sleep(2)
    time.sleep(0.5)
    print("S-responde")

def cliente(nombre, espera):
    time.sleep(espera)
    print(f"{nombre}-conecta")
    evento.set()
    time.sleep(0.5)
    with lock:
        print(f"{nombre}-envia")
        time.sleep(1)
    time.sleep(0.5)
    print(f"{nombre}-fin")

s = threading.Thread(target=servidor)
a = threading.Thread(target=cliente, args=("A", 1))
b = threading.Thread(target=cliente, args=("B", 2.8), daemon=True)
s.start(); a.start(); b.start()
s.join(); a.join()
print("main-fin")
```

**6.2) (1.5 puntos)** `donde()` retorna `"principal"` o `"secundario"` según el thread que la ejecuta. La ventana se muestra en `t = 0` y el usuario presiona una tecla en `t = 1,5`.

```python
class Descargador(QThread):
    senal_progreso = pyqtSignal(int)

    def run(self):
        print("[run]", donde())
        for i in range(3):
            time.sleep(1)
            self.senal_progreso.emit(i)

    def pausar(self):
        print("[pausar]", donde())
        time.sleep(2)


class Ventana(QWidget):
    senal_pausa = pyqtSignal()

    def __init__(self):
        super().__init__()
        self.descargador = Descargador()
        self.descargador.senal_progreso.connect(self.mostrar)
        self.senal_pausa.connect(self.descargador.pausar)
        self.descargador.start()

    def mostrar(self, i):
        print("[mostrar]", i, donde())

    def keyPressEvent(self, evento):
        self.senal_pausa.emit()
```

- Escribe el output e indica aproximadamente en qué instante aparece cada línea.
- ¿Se pausa la descarga? ¿Qué nota el usuario entre `t = 1,5` y `t = 3,5`?
- Corrige el diseño para que pausar funcione sin congelar la ventana.

**6.3) (1 punto)** ¿Qué imprime este programa? Justifica con una línea de tiempo. Luego indica qué cambiaría si en `__init__` se escribiera `timer = QTimer()` (variable local, sin padre, y las líneas siguientes usaran `timer` en vez de `self.timer`) en vez de `self.timer = QTimer(self)`.

```python
class Reloj(QWidget):
    def __init__(self):
        super().__init__()
        self.valor = 0
        self.timer = QTimer(self)
        self.timer.setInterval(1000)
        self.timer.timeout.connect(self.tick)
        self.timer.start()
        QTimer.singleShot(2500, self.timer.stop)
        QTimer.singleShot(4200, self.timer.start)
        QTimer.singleShot(7000, self.mostrar)

    def tick(self):
        self.valor += 1

    def mostrar(self):
        print(self.valor)
```

**6.4) (1.5 puntos)** Estas funciones deberían implementar el protocolo “4 bytes de largo en big endian + mensaje”. Encuentra **4 errores**, explica la consecuencia de cada uno y corrígelos.

```python
def enviar(sock, datos: bytes):
    sock.sendall(len(datos).to_bytes(4, "big"))
    enviados = 0
    while enviados < len(datos):
        enviados = sock.send(datos[enviados:])

def recibir(sock) -> bytes:
    largo = int.from_bytes(sock.recv(4), "little")
    return sock.recv(largo)
```

Si el emisor envía un mensaje de 300 bytes, ¿qué valor de `largo` calcula este receptor?